# Stage 03 / Lesson 09: Enterprise Cloud Vector Database: DataStax Astra DB
**Course Lecture:** Lecture 35: Working With Full Fledged DataStax Astra VectorDB  
**Section:** Section 7: Vector Stores and Vector Databases

---

## 🧭 Architectural Overview: From Vector Stores to Enterprise Vector Databases
In preceding lessons, we explored local embedded vector stores (ChromaDB, Meta FAISS, and InMemoryVectorStore).  
This lecture marks the critical architectural shift to **Cloud-Native Enterprise Vector Databases**, specifically **DataStax Astra DB**.

### Core Architecture & Technical Highlights:
1. **Engine Foundation (Apache Cassandra):**
   - Astra DB is engineered on Apache Cassandra, providing peer-to-peer distributed architecture, linear write scalability, and multi-region continuous availability.
   - Vector search is executed natively using advanced Vector Search indexing (VSI), supporting sub-millisecond retrieval across billions of embeddings.
2. **Serverless Vector Architecture:**
   - Decoupled compute and storage: Auto-scales down to zero and handles burst workloads dynamically without manual cluster provisioning.
3. **Multi-Modal Document Collections:**
   - Operates as a schemaless document database (JSON-like documents) with integrated vector columns, allowing full metadata filtering alongside vector distance calculations.
4. **Key Integration Parameters:**
   - **`ASTRA_DB_API_ENDPOINT`**: Cloud gateway endpoint (`https://<DB_ID>-<REGION>.apps.astra.datastax.com`).
   - **`ASTRA_DB_APPLICATION_TOKEN`**: High-security RBAC token (`AstraCS:...`).
   - **`ASTRA_DB_KEYSPACE`**: Keyspace / namespace (defaults to `default_keyspace`).

## 1. Setup, Environment & Connection Credentials
Initialize dependencies and verify configuration credentials.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# load environment variables
load_dotenv()

# read astra db credentials
astra_db_api_endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
astra_db_application_token = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

# check credential presence
has_credentials = bool(
    astra_db_api_endpoint 
    and astra_db_application_token 
    and not astra_db_api_endpoint.startswith("https://your-")
)

print("✅ Configuration checked.")
print(f"Credential status: {'LIVE ASTRA CLUSTER CONFIGURED' if has_credentials else 'DEMONSTRATION SIMULATOR ACTIVE'}")
if has_credentials:
    print(f"API Endpoint: {astra_db_api_endpoint[:35]}...")

✅ Configuration checked.
Credential status: DEMONSTRATION SIMULATOR ACTIVE


## 2. Embedding Model & Vector Store Initialization
Instantiate the embedding model and initialize `AstraDBVectorStore` via `langchain-astradb`.

In [2]:
# initialize embedding model (local HuggingFace with OpenAI fallback)
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAI Embeddings (text-embedding-3-small)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using local HuggingFace Embeddings (all-MiniLM-L6-v2)")

# connect to vector store
from langchain_astradb import AstraDBVectorStore

if has_credentials:
    vector_store = AstraDBVectorStore(
        collection_name="astra_vector_langchain",
        embedding=embeddings,
        api_endpoint=astra_db_api_endpoint,
        token=astra_db_application_token,
        namespace="default_keyspace"
    )
    print("✅ Connected to Live DataStax Astra DB!")
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embeddings)
    print("✅ Initialized enterprise vector store demonstration engine.")

Using local HuggingFace Embeddings (all-MiniLM-L6-v2)
✅ Initialized enterprise vector store demonstration engine.


## 3. Ingesting Multi-Domain Corpus with Rich Metadata
Construct a corpus containing 10 heterogeneous documents representing tweets, news bulletins, health articles, and technical documentation.

In [3]:
from langchain_core.documents import Document

# multi-domain corpus matching lecture curriculum
sample_documents = [
    Document(
        page_content="I had delicious chocolate chip pancakes and scrambled eggs for breakfast.",
        metadata={"source": "tweet", "topic": "food", "author": "foodie_99"}
    ),
    Document(
        page_content="Building an exciting enterprise generative AI project using LangChain and LangGraph.",
        metadata={"source": "tweet", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="The weather forecast for tomorrow indicates cloudy skies with afternoon rain showers.",
        metadata={"source": "news", "topic": "weather", "urgency": "low"}
    ),
    Document(
        page_content="Stock market indices surged today driven by strong quarterly earnings in tech sectors.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
    Document(
        page_content="A balanced breakfast containing healthy proteins and fruits helps maintain daytime focus.",
        metadata={"source": "health_blog", "topic": "food", "author": "nutritionist"}
    ),
    Document(
        page_content="Stealing from the bank or committing wire fraud is a serious federal crime.",
        metadata={"source": "news", "topic": "legal", "urgency": "high"}
    ),
    Document(
        page_content="DataStax Astra DB provides near-zero latency serverless vector database powered by Apache Cassandra.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "cloud_arch"}
    ),
    Document(
        page_content="Severe tropical storm warnings were issued along coastal regions with high wind speeds.",
        metadata={"source": "news", "topic": "weather", "urgency": "high"}
    ),
    Document(
        page_content="LangChain Expression Language (LCEL) simplifies building streaming conversational RAG pipelines.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="Central banks announced updated interest rate guidance following the economic policy summit.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
]

# ingest corpus
doc_ids = vector_store.add_documents(documents=sample_documents)

print(f"✅ Successfully ingested {len(doc_ids)} documents into the vector database.")
for idx, (doc_id, doc) in enumerate(zip(doc_ids[:3], sample_documents[:3]), start=1):
    print(f"  [{idx}] ID: {doc_id} | Source: {doc.metadata['source']:<10} | Content: '{doc.page_content[:50]}...'")

✅ Successfully ingested 10 documents into the vector database.
  [1] ID: 980e11da-b855-437a-9658-f9f45a5c6f5f | Source: tweet      | Content: 'I had delicious chocolate chip pancakes and scramb...'
  [2] ID: 5bd4de61-cfc8-4a71-b011-71c0f6fc1676 | Source: tweet      | Content: 'Building an exciting enterprise generative AI proj...'
  [3] ID: 4906dece-0921-42a9-b35b-adf2e9e3c743 | Source: news       | Content: 'The weather forecast for tomorrow indicates cloudy...'


## 4. Semantic Similarity Search
Perform semantic search across the collection.

In [4]:
# execute similarity search
weather_query = "What is the weather forecast for tomorrow?"
weather_results = vector_store.similarity_search(weather_query, k=3)

print(f"🔍 Query: '{weather_query}'\n")
print(f"Top {len(weather_results)} Retrieved Documents:")
for idx, doc in enumerate(weather_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source'):<10} | Topic: {doc.metadata.get('topic'):<8}")
    print(f"      Content: '{doc.page_content}'\n")

🔍 Query: 'What is the weather forecast for tomorrow?'

Top 3 Retrieved Documents:
  [1] Source: news       | Topic: weather 
      Content: 'The weather forecast for tomorrow indicates cloudy skies with afternoon rain showers.'

  [2] Source: news       | Topic: weather 
      Content: 'Severe tropical storm warnings were issued along coastal regions with high wind speeds.'

  [3] Source: health_blog | Topic: food    
      Content: 'A balanced breakfast containing healthy proteins and fruits helps maintain daytime focus.'


## 5. Native Metadata Filtering
Execute filtered queries where vector similarity search is restricted to specific metadata predicates (e.g., `source == 'tweet'`).

In [5]:
# search with metadata filter
tweet_query = "What projects are people building with LangChain?"

# define filter predicate (dict for AstraDB, callable for fallback engine)
filter_predicate = {"source": "tweet"} if has_credentials else (lambda doc: doc.metadata.get("source") == "tweet")

tweet_results = vector_store.similarity_search(
    query=tweet_query,
    k=2,
    filter=filter_predicate
)

print(f"🔍 Query: '{tweet_query}' (Filter: source == 'tweet')\n")
print(f"Filtered Results Count: {len(tweet_results)}")
for idx, doc in enumerate(tweet_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source')} | Author: {doc.metadata.get('author')}")
    print(f"      Content: '{doc.page_content}'\n")

🔍 Query: 'What projects are people building with LangChain?' (Filter: source == 'tweet')

Filtered Results Count: 2
  [1] Source: tweet | Author: dev_guru
      Content: 'Building an exciting enterprise generative AI project using LangChain and LangGraph.'

  [2] Source: tweet | Author: foodie_99
      Content: 'I had delicious chocolate chip pancakes and scrambled eggs for breakfast.'


## 6. Relevance Scoring Analysis
Retrieve document chunks paired with their calculated similarity confidence scores.

In [6]:
# query with confidence scoring
scored_results = vector_store.similarity_search_with_score("What legal consequences exist for bank theft?", k=2)

print("📊 Relevance Scored Results:")
for idx, (doc, score) in enumerate(scored_results, start=1):
    print(f"  Rank #{idx} | Score: {score:.4f}")
    print(f"    Topic:   {doc.metadata.get('topic')} | Source: {doc.metadata.get('source')}")
    print(f"    Content: '{doc.page_content}'\n")

📊 Relevance Scored Results:
  Rank #1 | Score: 0.6670
    Topic:   legal | Source: news
    Content: 'Stealing from the bank or committing wire fraud is a serious federal crime.'

  Rank #2 | Score: 0.0410
    Topic:   food | Source: tweet
    Content: 'I had delicious chocolate chip pancakes and scrambled eggs for breakfast.'


## 7. Advanced Retrievers: Score Threshold & Maximal Marginal Relevance (MMR)
Demonstrate advanced retriever patterns introduced in Lecture 35:
1. **`similarity_score_threshold`**: Discards chunks whose similarity score falls below a minimum confidence boundary.
2. **`mmr` (Maximal Marginal Relevance)**: Balances query relevance with diversity across results.

In [7]:
# 1. threshold retriever (supported in AstraDB)
try:
    threshold_retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 2, "score_threshold": 0.4}
    )
    test_query = "Is stealing from the bank a crime?"
    threshold_matches = threshold_retriever.invoke(test_query)
    print(f"🔍 Threshold Matches for '{test_query}': {len(threshold_matches)}")
    if threshold_matches:
        print(f"   Match: '{threshold_matches[0].page_content}'\n")
except Exception as e:
    print(f"ℹ️ similarity_score_threshold is an AstraDB-native capability: {e}\n")

# 2. mmr retriever
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5}
)

# invoke mmr retriever
mmr_matches = mmr_retriever.invoke("Tell me about breakfast options")
print(f"🔍 MMR Diverse Matches: {len(mmr_matches)}")
for idx, m in enumerate(mmr_matches, start=1):
    print(f"   [{idx}] {m.page_content}")

ℹ️ similarity_score_threshold is an AstraDB-native capability: 

🔍 MMR Diverse Matches: 2
   [1] A balanced breakfast containing healthy proteins and fruits helps maintain daytime focus.
   [2] I had delicious chocolate chip pancakes and scrambled eggs for breakfast.


## 8. End-to-End LCEL RAG Pipeline Integration
Connect the Astra DB retriever to a production-grade LCEL pipeline with Groq LLM inference.

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# formatting utility
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# context-grounded prompt
template = """You are an enterprise AI technical assistant.
Answer the user's question using ONLY the provided context. If the answer cannot be determined from the context, state "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# groq chat model
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# base retriever
base_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# lcel pipeline
rag_chain = (
    {"context": base_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# execute test query
rag_query = "What is DataStax Astra DB and what engine powers it?"
answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Answer:\n{answer}")

❓ Question: What is DataStax Astra DB and what engine powers it?

💬 Answer:
DataStax Astra DB is a near‑zero latency, serverless vector database, and it is powered by Apache Cassandra.


## 9. Architectural Comparison: Embedded Stores vs Cloud Vector Databases
| Architectural Aspect | Embedded Stores (Chroma / FAISS) | Cloud Vector Databases (DataStax Astra DB / Pinecone) |
| :--- | :--- | :--- |
| **Hosting & Topology** | Local application host (file-system / RAM) | Distributed multi-tenant or serverless cloud cluster |
| **Horizontal Scalability** | Limited to single node memory/disk | Practically unlimited (Auto-sharded, multi-region replication) |
| **High Availability SLA** | 0% (Depends entirely on host process uptime) | 99.95%+ Enterprise SLA with disaster recovery |
| **Concurrency & CRUD** | Single-writer locks (SQLite limitations) | Massive concurrent reads/writes powered by Cassandra / Raft |
| **Security & RBAC** | File-system permissions only | Enterprise Role-Based Access Control, Token management, SSO |
| **Primary Use-Case** | Local development, desktop tools, air-gapped apps | Production SaaS applications, live user-facing AI products |